# 08: Group-aware sampling and shortcut learning

![Group-aware pipeline](../images/08_group_aware_sampling.svg)

**Learning goals:** split independent groups, sample temporal windows reproducibly, apply consistent transforms, record lineage, and measure how group artifacts can inflate row-level evaluation. All data are synthetic.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f'NumPy {np.__version__}, synthetic data only')

## 1. Build repeated observations with acquisition artifacts

Each group receives a random binary label and a strong group-specific artifact vector. Repeated rows from a group share that artifact. The final coordinate contains a weak intended label signal. A nearest-neighbor model can exploit group identity whenever the same groups occur in training and test data.

In [ ]:
n_groups, repeats, artifact_dim = 60, 8, 10
group_labels = rng.integers(0, 2, size=n_groups)
artifacts = rng.normal(0, 4.0, size=(n_groups, artifact_dim))
groups = np.repeat(np.arange(n_groups), repeats)
labels = group_labels[groups]
artifact_features = artifacts[groups] + rng.normal(0, 0.15, size=(len(groups), artifact_dim))
intended = (2 * labels - 1)[:, None] * 0.25 + rng.normal(0, 1.0, size=(len(groups), 1))
features = np.concatenate([artifact_features, intended], axis=1)
print(f'features={features.shape}, groups={np.unique(groups).size}, rows/group={repeats}')
assert features.shape == (n_groups * repeats, artifact_dim + 1)

## 2. Compare a row split with a group-disjoint split

`train_test_split` knows nothing about groups. `GroupShuffleSplit` assigns all rows with one group key together. The test below asserts the defining invariant: the two group sets have an empty intersection.

In [ ]:
indices = np.arange(len(features))
row_train, row_test = train_test_split(indices, test_size=0.3, random_state=SEED, stratify=labels)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
group_train, group_test = next(splitter.split(features, labels, groups=groups))

row_overlap = set(groups[row_train]) & set(groups[row_test])
group_overlap = set(groups[group_train]) & set(groups[group_test])
print(f'row split shared groups={len(row_overlap)}; group split shared groups={len(group_overlap)}')
assert len(group_overlap) == 0
assert len(row_overlap) > 0

In [ ]:
def fit_score(train_idx, test_idx, use_artifacts=True):
    columns = slice(None) if use_artifacts else [-1]
    model = KNeighborsClassifier(n_neighbors=1)
    model.fit(features[train_idx][:, columns], labels[train_idx])
    return accuracy_score(labels[test_idx], model.predict(features[test_idx][:, columns]))

scores = {
    'row split, all features': fit_score(row_train, row_test),
    'group split, all features': fit_score(group_train, group_test),
    'group split, intended only': fit_score(group_train, group_test, use_artifacts=False),
}
for name, score in scores.items():
    print(f'{name:30s}: {score:.3f}')
assert scores['row split, all features'] > scores['group split, all features'] + 0.2

The score gap is the lesson, not a universal numerical threshold. The row split rewards recognition of group artifacts. Group-disjoint evaluation asks whether the rule transfers to unseen groups. An artifact-only baseline, subgroup scores, and counterfactual artifact changes are complementary diagnostics.

## 3. Deterministic and stochastic windows

Python slices use a half-open interval `[start:start + window]`. Deterministic starts support stable evaluation. A passed `numpy.random.Generator` makes stochastic training choices reproducible without relying on global state.

In [ ]:
def deterministic_starts(length, window, stride, include_endpoint=True):
    if not (0 < window <= length) or stride <= 0:
        raise ValueError('require 0 < window <= length and positive stride')
    starts = list(range(0, length - window + 1, stride))
    endpoint = length - window
    if include_endpoint and starts[-1] != endpoint:
        starts.append(endpoint)
    return starts

def stochastic_start(length, window, generator):
    if not 0 < window <= length:
        raise ValueError('require 0 < window <= length')
    return int(generator.integers(0, length - window + 1))

starts = deterministic_starts(length=11, window=4, stride=3)
draws = [stochastic_start(11, 4, rng) for _ in range(5)]
print('deterministic:', starts, 'stochastic:', draws)
assert starts == [0, 3, 6, 7]

## 4. One transform draw per temporal window

A horizontal flip uses one Bernoulli draw for all frames. `np.flip` can produce negative strides, so `.copy()` makes memory contiguous and compatible with tensor conversion.

In [ ]:
def transform_window(window, generator):
    flip = bool(generator.random() < 0.5)
    transformed = np.flip(window, axis=2).copy() if flip else window.copy()
    return transformed, {'horizontal_flip': flip}

sequence = np.arange(12 * 3 * 4).reshape(12, 3, 4)
start, width = 3, 5
window = sequence[start:start + width]
transformed, transform_record = transform_window(window, rng)
lineage = {
    'source_id': 'synthetic-sequence-0042', 'group_id': 12, 'split': 'train',
    'start': start, 'stop': start + width, 'preprocess_version': 'v1', **transform_record,
}
print(lineage)
assert transformed.shape == window.shape and transformed.flags['C_CONTIGUOUS']

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8), constrained_layout=True)
ax.barh(list(scores), list(scores.values()), color=['#c75b4b', '#367cad', '#27896f'])
ax.set(xlim=(0, 1), xlabel='accuracy', title='A row split rewards the group shortcut')
ax.axvline(0.5, color='black', linestyle=':', linewidth=1)
plt.show()

## Exercises and takeaways

1. Set `include_endpoint=False`. **Check:** starts become `[0, 3, 6]`, so the final possible offset is not evaluated.
2. Group by each row instead of the true group. **Check:** the disjointness assertion passes formally but no longer protects the independent unit.
3. Add `device_id` to the lineage dictionary. **Check:** it enables artifact-only and per-device evaluation later.

**Efficiency:** `np.isin` forms group masks without row loops, index manifests avoid storing duplicate windows, and view-based window functions save memory if callers avoid unsafe in-place writes.

**Takeaways:** split independent groups before creating windows; fix evaluation windows; sample training windows with explicit generators; share plausible transform parameters through time; and retain lineage so suspicious performance can be traced to its source.

## Continue learning

[Previous notebook: 07](07_gradient_updates_and_schedules.ipynb) | [Lecture](../lectures/08_group_aware_sampling.md) | [Curriculum](../README.md) | [Next notebook: 09](09_eigenspectra_and_effective_rank.ipynb)